# Capítulo 12 · Laboratório 12: fôlego de mercado (Market Breadth) na B3
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**, com 72 ações que estão hoje no Ibovespa. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap12.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Os indicadores de fôlego escritos à mão, com pandas: Linha AD, TRIN em reais, McClellan e Summation (clássicos e com razão ajustada), novas máximas e mínimas, High-Low Index, % de ações acima da MM200, pivôs em massa e a concentração do índice (HHI). O laboratório testa o general sem tropas, a mudança brusca do McClellan, os impulsos de Zweig, SuperZweig e Deemer e o filtro de fôlego do Método 6C. Atenção: a lista de hoje cria viés de sobrevivência, e o valor de mercado do HHI é o do dia em que a cópia local foi criada.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap12.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=1: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ACOES = ["ABEV3", "ALOS3", "ASAI3", "AZZA3", "B3SA3", "BBAS3", "BBDC3", "BBDC4", "BBSE3", "BEEF3", "BPAC11",
         "BRAP4", "BRAV3", "MBRF3", "CMIG4", "CMIN3", "CPFE3", "CPLE3", "CSAN3", "CSNA3", "CXSE3", "CYRE3",
         "DIRR3", "EGIE3", "AXIA3", "EMBJ3", "ENEV3", "ENGI11", "EQTL3", "FLRY3", "GGBR4", "GOAU4", "HAPV3",
         "HYPE3", "IGTI11", "IRBR3", "ISAE4", "ITSA4", "ITUB4", "KLBN11", "LREN3", "MGLU3", "MOTV3", "MRVE3",
         "MULT3", "NATU3", "PCAR3", "PETR3", "PETR4", "PRIO3", "PSSA3", "RADL3", "RAIL3", "RAIZ4", "RDOR3",
         "RENT3", "SANB11", "SBSP3", "SLCE3", "SMFT3", "SUZB3", "TAEE11", "TIMS3", "TOTS3", "UGPA3", "USIM5",
         "VALE3", "VBBR3", "VIVA3", "VIVT3", "WEGE3", "YDUQ3"]
HOLDINGS = ("PETR3", "BBDC3", "ITSA4", "BRAP4", "GOAU4")     # fora do HHI: 1 ação por empresa, sem holdings


def baixa(t, ini="2008-01-01"):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
if "^BVSP" not in D:
    D["^BVSP"] = baixa("^BVSP")
confere({"^BVSP": D["^BVSP"]})
for t in ACOES:
    if t not in D:
        D[t] = baixa(f"{t}.SA")
if "caps" not in D:                     # valor de mercado do dia em que o cache foi criado (retrato, não série)
    caps = {}
    for t in ACOES:
        if t in HOLDINGS:
            continue
        try:
            caps[t] = yf.Ticker(f"{t}.SA").fast_info["market_cap"]
        except Exception:
            pass
    D["caps"] = pd.Series(caps, dtype=float)
pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
def limpa(d):
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"]) & (d["Volume"] > 0)
    return d[ok]


IB = D["^BVSP"]["Close"]
X = {t: limpa(D[t]) for t in ACOES if len(D[t])}
fech = pd.DataFrame({t: d["Close"] for t, d in X.items()}).reindex(IB.index)
maxi = pd.DataFrame({t: d["High"] for t, d in X.items()}).reindex(IB.index)
mini = pd.DataFrame({t: d["Low"] for t, d in X.items()}).reindex(IB.index)
vfin = pd.DataFrame({t: d["Close"] * d["Volume"] for t, d in X.items()}).reindex(IB.index)   # volume em R$
print(f"Parte 1. {len(X)} ações (de {len(ACOES)}); {int(fech.notna().iloc[-1].sum())} com negócio em"
      f" {IB.index[-1]:%d/%m/%Y}; {int(fech.notna().iloc[0].sum())} já existiam em {IB.index[0]:%d/%m/%Y}.")

## Parte 2: os indicadores

In [ ]:
var = fech.diff()
A, Dc = (var > 0).sum(axis=1), (var < 0).sum(axis=1)
OK = (A + Dc) >= 20                                       # só dias com amostra decente
A, Dc = A[OK], Dc[OK]
LIQ = A - Dc                                              # avanço líquido
LINHA_AD = LIQ.cumsum()
VA, VD = vfin.where(var > 0).sum(axis=1)[OK], vfin.where(var < 0).sum(axis=1)[OK]
TRIN = (A / Dc) / (VA / VD)
TRIN = TRIN.replace([np.inf, -np.inf], np.nan)


def mcclellan(x):
    return x.ewm(alpha=0.10, adjust=False).mean() - x.ewm(alpha=0.05, adjust=False).mean()   # MME de 19 menos MME de 39


RANA = LIQ / (A + Dc) * 1000                              # razão ajustada (× 1.000)
MC_CLAS, MC_AJ = mcclellan(LIQ), mcclellan(RANA)
SUM_CLAS = 1000 + MC_CLAS.cumsum()                        # tradicional: neutro em +1.000
RASI = MC_AJ.cumsum()                                     # razão ajustada: oscila em torno de zero
RAZAO = A / (A + Dc)
ZW = RAZAO.ewm(span=10, adjust=False).mean()              # Zweig: MME de 10 de avanços / (avanços + declínios)
print(f"\nParte 2. Último pregão: {A.iloc[-1]} em alta, {Dc.iloc[-1]} em baixa | TRIN {nr(TRIN.iloc[-1])} | McClellan"
      f" clássico {nr(MC_CLAS.iloc[-1], 1)} e ajustado {nr(MC_AJ.iloc[-1], 0)} | Summation {nr(SUM_CLAS.iloc[-1], 0)} |"
      f" RASI {nr(RASI.iloc[-1], 0)} | Zweig {nr(ZW.iloc[-1])}")

## Parte 3: as contas do texto

In [ ]:
print(f"\nParte 3. Exemplo do texto: 235 em alta e 255 em baixa dão avanço líquido de {235 - 255}.")
print(f"   Fatores das médias: 2 / (19 + 1) = {nr(2 / 20)} e 2 / (39 + 1) = {nr(2 / 40)}.")
print(f"   TRIN de exemplo: 300 altas, 200 baixas, R$ 6 bi em alta e R$ 3 bi em baixa = (300/200) / (6/3) ="
      f" {nr((300 / 200) / (6 / 3))} (abaixo de 1: o volume de alta pesou mais que o número de altas).")


def fwd(h):
    return IB.shift(-h) / IB - 1


def resumo(datas, h):
    r = fwd(h).reindex(datas).dropna()
    return len(r), (r.mean() if len(r) else np.nan), ((r > 0).mean() if len(r) else np.nan)


def primeiros(cond, espaco=20):
    """Primeiro dia de cada sequência; ignora novos sinais a menos de 'espaco' pregões do anterior."""
    dias, ult = [], None
    for i, (dt, v) in enumerate(cond.items()):
        if v and (ult is None or i - ult > espaco):
            dias.append(dt)
            ult = i
    return pd.DatetimeIndex(dias)

## Parte 4: TRIN

In [ ]:
T10 = TRIN.rolling(10).mean()
TR = {}
print("\nParte 4. TRIN em R$ (média de 10) desde 2008: retorno do Ibovespa nos 20 pregões seguintes")
for nome, cond in (("abaixo de 0,8 (sobrecomprado)", T10 < 0.8), ("acima de 1,2 (sobrevendido)", T10 > 1.2),
                   ("entre 0,8 e 1,2", (T10 >= 0.8) & (T10 <= 1.2))):
    n, m, p = resumo(cond[cond].index, 20)
    TR[nome] = (cond.mean(), n, m, p)
    print(f"   {nome:<30} {pc(cond.mean(), 0):>4} dos dias | retorno médio {ps(m, 2)} | positivos {pc(p, 0)}")

## Parte 5: o general sem tropas

In [ ]:
topo = IB >= IB.rolling(252).max()
ad_topo = LINHA_AD >= LINHA_AD.rolling(252).max()
MM200 = fech.rolling(200, min_periods=150).mean()      # aceita alguns dias sem negócio na janela
DEN = (MM200.notna() & fech.notna()).sum(axis=1)
ACIMA200 = ((fech > MM200).sum(axis=1) / DEN * 100).where(DEN >= 20)   # dias com poucas ações não contam
GEN = {}
print("\nParte 5. Ibovespa em máxima de 52 semanas: o fôlego acompanhava? Retorno nos 60 pregões seguintes")
for nome, cond in (("Linha AD também na máxima", topo & ad_topo.reindex(IB.index, fill_value=False)),
                   ("Linha AD abaixo da máxima", topo & ~ad_topo.reindex(IB.index, fill_value=True))):
    n, m, p = resumo(cond[cond].index, 60)
    GEN[nome] = (n, m, p)
    print(f"   {nome:<28} {n:>4} dias | retorno médio {ps(m, 2)} | positivos {pc(p, 0)}")
n, m, p = resumo(IB.index, 60)
print(f"   {'qualquer dia':<28} {n:>4} dias | retorno médio {ps(m, 2)} | positivos {pc(p, 0)}")
GEN["qualquer dia"] = (n, m, p)

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
w = IB.loc["2023":]
axs[0].plot(w.index, w, color="black")
axs[0].set_title("Ibovespa")
axs[1].plot(LINHA_AD.loc["2023":], color="gray")
axs[1].set_title("Linha AD das 72 ações")
plt.show()

## Parte 6: mudança brusca do McClellan

In [ ]:
abaixo = (MC_AJ < -50).rolling(10).max().astype(bool)
brusca = primeiros((MC_AJ > 50) & abaixo.shift().fillna(False).astype(bool))
q = MC_AJ.quantile([0.05, 0.95])
BR = {"sinais": brusca, "q": q}
print(f"\nParte 6. McClellan ajustado na B3: 90% dos dias entre {nr(q.iloc[0], 0)} e {nr(q.iloc[1], 0)}.")
print(f"   Mudanças bruscas (de abaixo de -50 para acima de +50 em até 10 pregões): {len(brusca)} desde 2008")
for h in (20, 60, 120):
    n, m, p = resumo(brusca, h)
    BR[h] = (n, m, p)
    print(f"   Ibovespa em {h:>3} pregões: média {ps(m)} | positivos {pc(p, 0)} ({n} sinais)")

## Parte 7: impulsos de fôlego

In [ ]:
def thrusts(serie, baixo, alto, janela):
    """Datas em que a série vai de abaixo de 'baixo' para acima de 'alto' em até 'janela' pregões."""
    sinais, ult_baixo = [], None
    for i, (dt, v) in enumerate(serie.items()):
        if v < baixo:
            ult_baixo = i
        elif v > alto and ult_baixo is not None and i - ult_baixo <= janela:
            sinais.append(dt)
            ult_baixo = None
    return pd.DatetimeIndex(sinais)


deemer = A.rolling(10).sum() / Dc.rolling(10).sum()
IMP = {"Zweig (0,40 para 0,615 em 10 pregões)": thrusts(ZW, 0.40, 0.615, 10),
       "SuperZweig (0,35 para 0,64 em 30 pregões)": thrusts(ZW, 0.35, 0.64, 30),
       "Breakaway Momentum de Deemer (1,97)": primeiros(deemer >= 1.97, 60)}
IMPR = {}
print("\nParte 7. Impulsos de fôlego desde 2008 e o Ibovespa depois")
for nome, s in IMP.items():
    IMPR[nome] = {h: resumo(s, h) for h in (20, 60, 120)}
    print(f"   {nome}: {len(s)} sinais ({', '.join(f'{x:%m/%Y}' for x in s)})")
    print("      " + " | ".join(f"{h} pregões: {ps(IMPR[nome][h][1])} ({pc(IMPR[nome][h][2], 0)} positivos)" for h in (20, 60, 120)))
IMPR["qualquer dia"] = {h: resumo(IB.index, h) for h in (20, 60, 120)}
print("   qualquer dia: " + " | ".join(f"{h} pregões: {ps(IMPR['qualquer dia'][h][1])} ({pc(IMPR['qualquer dia'][h][2], 0)}"
                                    " positivos)" for h in (20, 60, 120)))

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axs[0].semilogy(IB.index, IB, color="black")
for x in IMP["Zweig (0,40 para 0,615 em 10 pregões)"]:
    axs[0].plot(x, IB[x] * 0.9, "^", color="black")
axs[1].plot(ZW, lw=0.5)
for y in (0.40, 0.615):
    axs[1].axhline(y, color="gray", ls="--")
plt.show()

## Parte 8: máximas, mínimas, High-Low Index e % acima da MM200

In [ ]:
NMAX = (maxi >= maxi.rolling(252, min_periods=200).max()).sum(axis=1)
NMIN = (mini <= mini.rolling(252, min_periods=200).min()).sum(axis=1)
NMAX, NMIN = NMAX.where(OK.reindex(NMAX.index, fill_value=False)), NMIN.where(OK.reindex(NMIN.index, fill_value=False))
HLI = (NMAX / (NMAX + NMIN).replace(0, np.nan) * 100).ffill().rolling(10).mean()
F6C = {}
print("\nParte 8. Filtro de contexto do Método 6C: % de ações acima da MM200 e o Ibovespa nos 60 pregões seguintes")
for nome, cond in (("acima de 50%", ACIMA200 > 50), ("até 50%", ACIMA200 <= 50)):
    c = cond & ACIMA200.notna() & (IB.index >= "2009-01-01")
    n, m, p = resumo(c[c].index, 60)
    F6C[nome] = (c.mean(), n, m, p)
    print(f"   {nome:<13} {n:>5} dias | retorno médio {ps(m, 2)} | positivos {pc(p, 0)}")
print(f"   High-Low Index (média de 10) abaixo de 30: retorno médio em 60 pregões"
      f" {ps(resumo(HLI[HLI < 30].index, 60)[1], 2)}; acima de 70: {ps(resumo(HLI[HLI > 70].index, 60)[1], 2)}")

## Parte 9: pivôs em massa

In [ ]:
N = 10  # versão didática: topo (fundo) mais alto (baixo) que as N barras de cada lado; só se confirma N pregões depois
PA = (maxi == maxi.rolling(2 * N + 1, center=True).max()) & maxi.notna()
PB = (mini == mini.rolling(2 * N + 1, center=True).min()) & mini.notna()
SALDO = (PA.sum(axis=1) - PB.sum(axis=1)).rolling(20).sum()
ULT = maxi.index[-N - 1]                                   # último dia em que um pivô já pode estar confirmado
HIST = SALDO.loc[:ULT].dropna()
print(f"\nParte 9. Pivôs (N = {N}) nos 20 pregões até {ULT:%d/%m/%Y}: {int(PA.loc[:ULT].iloc[-20:].sum().sum())} de alta e"
      f" {int(PB.loc[:ULT].iloc[-20:].sum().sum())} de baixa; saldo {HIST.iloc[-1]:+.0f}, maior que"
      f" {pc((HIST < HIST.iloc[-1]).mean(), 0)} dos dias desde 2008 (faixa de {HIST.min():+.0f} a {HIST.max():+.0f}).")

## Parte 10: concentração (HHI)

In [ ]:
caps = D["caps"].dropna()
caps = caps[caps > 0]
PESOS = (caps / caps.sum()).sort_values(ascending=False)
HHI = (PESOS ** 2).sum()
print(f"\nParte 10. HHI pelo valor de mercado de {len(caps)} empresas (aproximação; o Ibovespa usa free float e limites):"
      f" {nr(HHI, 3)}, equivale a {1 / HHI:.0f} ações de mesmo peso.")
print("   5 maiores pesos: " + ", ".join(f"{k} {pc(v)}" for k, v in PESOS.head(5).items()))

## Parte 11: o retrato do último pregão

In [ ]:
print(f"\nParte 11. Em {IB.index[-1]:%d/%m/%Y}: {NMAX.iloc[-1]:.0f} ações em máxima de 52 semanas e {NMIN.iloc[-1]:.0f} em mínima;"
      f" High-Low Index {nr(HLI.iloc[-1], 0)}; {nr(ACIMA200.iloc[-1], 0)}% acima da MM200; Ibovespa"
      f" {nr(IB.iloc[-1], 0)} pontos ({pc(IB.iloc[-1] / IB.rolling(252).max().iloc[-1] - 1)} da máxima de 52 semanas).")

## Parte 12: zonas extremas de fôlego

Pela experiência do autor, 80% ou mais das ações acima da média marcam topos, mesmo que intermediários, e 15% ou menos, fundos. O teste mede, a cada entrada na zona, se o Ibovespa caiu (ou subiu) 5% ou mais nos 40 pregões seguintes, nas médias de 8, 21, 50 e 200 pregões.

In [ ]:
# Parte 12: zonas extremas. Pela experiência do autor, 80% ou mais das ações acima da média marcam topos (mesmo que
# intermediários), e 15% ou menos, fundos. Teste: a cada entrada na zona, o Ibovespa caiu 5% ou mais (topo) ou subiu
# 5% ou mais (fundo) nos 40 pregões seguintes? Compara com um dia qualquer, desde 2009.
mn40 = IB[::-1].rolling(40, min_periods=1).min()[::-1].shift(-1) / IB - 1      # pior ponto dos 40 pregões seguintes
mx40 = IB[::-1].rolling(40, min_periods=1).max()[::-1].shift(-1) / IB - 1      # melhor ponto
validos = (IB.index >= "2009-01-01") & (np.arange(len(IB)) < len(IB) - 60)
QUEDA, ALTA = (mn40 <= -0.05)[validos], (mx40 >= 0.05)[validos]
print("\nParte 12. Zonas extremas de fôlego: o Ibovespa caiu (subiu) 5% ou mais nos 40 pregões seguintes?")
print(f"   dia qualquer: queda de 5% em {pc(QUEDA.mean(), 0)} | alta de 5% em {pc(ALTA.mean(), 0)}")
ZONAS = {}
for n in (8, 21, 50, 200):
    mm = fech.rolling(n, min_periods=max(int(n * 0.75), 5)).mean()
    den = (mm.notna() & fech.notna()).sum(axis=1)
    pct = ((fech > mm).sum(axis=1) / den * 100).where(den >= 20)
    linha = []
    for lim, topo in ((80, True), (85, True), (15, False), (10, False)):
        cond = (pct >= lim) if topo else (pct <= lim)
        entra = cond & ~cond.shift(1, fill_value=False)                           # primeiro dia de cada entrada
        dias = entra[entra & validos].index
        acerto = (QUEDA if topo else ALTA).reindex(dias).mean()
        ZONAS[(n, lim)] = (len(dias), acerto, resumo(dias, 20)[1], resumo(dias, 60)[1])
        linha.append(f"{'≥' if topo else '≤'}{lim}%: {pc(acerto, 0)} ({len(dias)})")
    print(f"   média de {n:>3}: " + " | ".join(linha))
print("   (≥80% e ≥85%: queda de 5%; ≤15% e ≤10%: alta de 5%; entre parênteses, as entradas na zona)")
for n, lim in ((200, 80), (200, 85), (200, 15), (200, 10)):
    k, a, r20, r60 = ZONAS[(n, lim)]
    print(f"   média de 200, zona {lim}%: retorno médio em 20 pregões {ps(r20, 2)}; em 60, {ps(r60, 2)}")
print(f"   dia qualquer: {ps(resumo(IB.index[validos], 20)[1], 2)} em 20 pregões; {ps(resumo(IB.index[validos], 60)[1], 2)} em 60")


## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, o trecho da Parte 2 que calcula `A`, `Dc`, `LIQ`, `LINHA_AD`, `VA`, `VD` e `TRIN`. Mostre por que o volume vem de `vfin` (volume em reais) e por que só entram os dias com `OK`." Confira com a Parte 3, que calcula um TRIN de exemplo (300 altas, 200 baixas, R$ 6 bi em alta e R$ 3 bi em baixa dão 0,75), e com a Parte 4, que imprime a média de 10 dias do TRIN abaixo de 0,8 em 6% dos dias e acima de 1,2 em 32%.
2. **Adaptar:** "Troque a lista `ACOES` pelas ações de [ÍNDICE], o código `"^BVSP"` pelo ticker de [ÍNDICE] no Yahoo Finance, em todas as vezes que ele aparece, e o nome do arquivo em `CACHE`, para não misturar com os dados do livro. Não mude nenhuma fórmula. A lista precisa ter pelo menos 20 ações, por causa do filtro `OK`." Antes de trocar, rode o caso do livro e confira se a Parte 5 mostra o Ibovespa com +1,76% em 60 pregões quando a Linha AD estava na máxima e +0,17% quando não estava, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 4, procure os níveis de TRIN que separam os 10% de dias mais altos e mais baixos na B3. O retorno muda?
2. Na Parte 7, recalibre o Zweig para a B3 (por exemplo, 0,35 e 0,65). Os sinais ficam melhores ou só mais raros?
3. Na Parte 9, troque N = 10 por N = 5. O saldo de pivôs fica mais rápido e mais ruidoso?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.